In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)

import torch
import numpy as np

from nocode_robot_programming.state_decision.state_decider import StateDeciderBase, DINOFeaturePresence, DINOFeaturePresenceConcat, DINOFeaturePresenceAttnGated, DINOWithMIL, StateDeciderSIFT, AEGP
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies
from gesture_detector.utils import pretty_confusion_matrix

seed = 48
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
dataset_builder = TrajectoryDatasetEvaluationViewBuilder()
datasets = dataset_builder.load_eval_from_task("petr_kin_peg_pick")

In [ ]:
dataset_builder.all_tasks

# State Estimation setup

In [ ]:
# I made some filter, but I don't like it like this:
tasktemplates_to_evaluate = ['peg_pick', 'probe', 'wrap']
tasks_to_evaluate = []
for t in dataset_builder.all_tasks:
    for tt in tasktemplates_to_evaluate:
        if (tt in t):
            tasks_to_evaluate.append(t) 
tasks_to_evaluate

In [ ]:
train_stats, test_stats, model_names = [], [], []
for decider in [
        DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=None), # best model
        DINOFeaturePresence(dino_variant= "facebook/dinov3-vits16-pretrain-lvd1689m", percentile_keep=None), # DINOv3
        DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=None),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=None, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),
        # DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=None, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=None, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        # DINOFeaturePresence(percentile_keep=0.0, dino_variant="dinov2_vitg14"),
        # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percent_keep=0),
        StateDeciderSIFT(method = "SIFT"), # suggesting the background problem
        StateDeciderSIFT(method = "AKAZE"),
        StateDeciderSIFT(method = "ORB"),
        AEGP(binary=False, pix=224),
        AEGP(binary=False, pix=224, crop=True),
    ]:
    train_stats_model, test_stats_model = [], []
    dataset_names = []
    for name in tasks_to_evaluate:
        
        for d_train, d_test, d_text in dataset_builder.load_eval_from_task(name, anomaly=False):
            print(f"Model: {decider.__class__.__name__}, Dataset: {d_text}")
            
            decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

            y_pred = decider.predict_many(d_train.X)
            # pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
            train_stats_model.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            # pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
            test_stats_model.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            # ipt.show(small=True)
            # ipt.delete() # don't plot
            dataset_names.append(d_text.split(",")[0])

    train_stats.append(train_stats_model)
    test_stats.append(test_stats_model)
    model_names.append(decider.__str__())

In [ ]:
TO_NICE_NAMES = { # complicated name -> nice name
'dinov2_vits14,224,mean': 'dinov2 small mean',
'facebook/dinov3-vits16-pretrain-lvd1689m,224,mean': 'dinov3 small mean',
'facebook/dinov3-vitl16-pretrain-lvd1689m,224,mean': 'dinov3 large mean',
'dinov2_vits14,224,concat': 'dinov2 small concat',
'dinov2_vits14,224,attn,hard,mean,0.4': 'dinov2 small attn',
'dinov2_vits14,224,MIL,H=128,e=1000': 'dinov2 small MIL',
'SIFT': "SIFT",
'AEGP,bin=False': 'AEGP Multiclass',
}

for i in range(len(model_names)):
    if model_names[i] in TO_NICE_NAMES:
        model_names[i] = TO_NICE_NAMES[model_names[i]]

model_names

In [ ]:
visualize_accuracies(100 * np.array(train_stats), 100 * np.array(test_stats), model_names, dataset_names, out_dir="plot", jupyter_plot=False)
f"Overall accuracy on train data: {round(100 * np.array(train_stats).mean())}% and test data {round(100 * np.array(test_stats).mean())}%"

# Anomaly detection setup

In [ ]:
train_stats, test_stats, model_names = [], [], []
for decider in [
        # DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=0.1), # best model
        # DINOFeaturePresence(dino_variant= "facebook/dinov3-vits16-pretrain-lvd1689m", percentile_keep=0.1), # DINOv3
        # DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=0.1),
        # DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.1, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),
        # # DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.1, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        # DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=0.1, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        # # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percentile_keep=0.1),
        # StateDeciderSIFT(method = "SIFT"), # suggesting the background problem
        # # StateDeciderSIFT(method = "AKAZE"),
        # # StateDeciderSIFT(method = "ORB"),
        AEGP(binary=False, pix=64),
        AEGP(binary=True, pix=64),
        # AEGP(binary=False, pix=64, crop=False),
        # AEGP(binary=True, pix=64, crop=False),
    ]:
    train_stats_model, test_stats_model = [], []
    dataset_names = []
    for name in dataset_builder.tasks.keys():
        
        for d_train, d_test, d_text in dataset_builder.load_eval_from_task(name, anomaly=True):
            print(f"Model: {decider.__class__.__name__}, Dataset: {d_text}")
            
            decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

            y_pred = decider.predict_many(d_train.X)
            # pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
            train_stats_model.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            # pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
            test_stats_model.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            # ipt.show(small=True)
            ipt.delete() # don't plot
            dataset_names.append(d_text.split(",")[0])

    train_stats.append(train_stats_model)
    test_stats.append(test_stats_model)
    model_names.append(decider.__str__())

In [ ]:
visualize_accuracies(100 * np.array(train_stats), 100 * np.array(test_stats), model_names, dataset_names, out_dir="plot", jupyter_plot=False)
f"Overall accuracy on train data: {round(100 * np.array(train_stats).mean())}% and test data {round(100 * np.array(test_stats).mean())}%"